<a href="https://colab.research.google.com/github/neohack22/ebw3nt/blob/main/apprentissage/correction_regularisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installez (ou mettez à jour) les bibliothèques nécessaires pour ce notebook, à savoir `keras` et `keras-hub`, en utilisant `pip` (en mode silencieux si possible).

In [ ]:
!pip install keras keras-hub --upgrade -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 60.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
keras-nlp 0.21.1 requires keras-hub==0.21.1, but you have keras-hub 0.26.0 which is incompatible.


Configurez Keras pour utiliser le backend **tensorflow** en définissant la variable d’environnement `KERAS_BACKEND` avant d’importer/initialiser Keras.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

In [ ]:
# @title
import os
from IPython.core.magic import register_cell_magic

@register_cell_magic
def backend(line, cell):
    current, required = os.environ.get("KERAS_BACKEND", ""), line.split()[-1]
    if current == required:
        get_ipython().run_cell(cell)
    else:
        print(
            f"This cell requires the {required} backend. To run it, change KERAS_BACKEND to "
            f"\"{required}\" at the top of the notebook, restart the runtime, and rerun the notebook."
        )

Chargez le dataset **MNIST**, puis préparez les images pour un réseau dense :
- aplatissez chaque image 28×28 en un vecteur de taille 784,
- convertissez en `float32` et normalisez entre 0 et 1.

Ensuite, construisez deux jeux d’entrées :
1. une version où vous concaténez **784 variables aléatoires** (canaux de bruit) aux 784 pixels,
2. une version où vous concaténez **784 zéros** aux 784 pixels.

In [ ]:
from keras.datasets import mnist
import numpy as np

(train_images, train_labels), _ = mnist.load_data()
train_images = train_images.reshape((60000, 28 * 28))
train_images = train_images.astype("float32") / 255

train_images_with_noise_channels = np.concatenate(
    [train_images, np.random.random((len(train_images), 784))], axis=1
)

train_images_with_zeros_channels = np.concatenate(
    [train_images, np.zeros((len(train_images), 784))], axis=1
)

Écrivez une fonction `get_model()` qui construit et compile un modèle `keras.Sequential` :
- Dense(512, activation="relu")
- Dense(10, activation="softmax")
- optimizer = `"adam"`
- loss = `"sparse_categorical_crossentropy"`
- metric = `"accuracy"`

Entraînez ensuite **deux fois** le modèle (10 époques, batch_size=128, validation_split=0.2) :
1. sur les images avec canaux de bruit,
2. sur les images avec canaux de zéros.

Stockez les historiques d’entraînement pour pouvoir comparer les performances.

In [ ]:
import keras
from keras import layers

def get_model():
    model = keras.Sequential(
        [
            layers.Dense(512, activation="relu"),
            layers.Dense(10, activation="softmax"),
        ]
    )
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

model = get_model()
history_noise = model.fit(
    train_images_with_noise_channels,
    train_labels,
    epochs=10,
    batch_size=128,
    validation_split=0.2,
)

model = get_model()
history_zeros = model.fit(
    train_images_with_zeros_channels,
    train_labels,
    epochs=10,
    batch_size=128,
    validation_split=0.2,
)

À partir des objets `history` obtenus, tracez sur un même graphique l’évolution de la **validation accuracy** sur 10 époques :
- courbe pour le modèle entraîné avec canaux de bruit,
- courbe pour le modèle entraîné avec canaux de zéros.

Ajoutez un titre, des labels d’axes, des ticks 1→10, une légende, puis affichez la figure.

In [ ]:
import matplotlib.pyplot as plt

val_acc_noise = history_noise.history["val_accuracy"]
val_acc_zeros = history_zeros.history["val_accuracy"]
epochs = range(1, 11)
plt.plot(
    epochs,
    val_acc_noise,
    "b-",
    label="Validation accuracy with noise channels",
)
plt.plot(
    epochs,
    val_acc_zeros,
    "r--",
    label="Validation accuracy with zeros channels",
)
plt.title("Effect of noise channels on validation accuracy")
plt.xlabel("Epochs")
plt.xticks(epochs)
plt.ylabel("Accuracy")
plt.legend()
plt.show()

Rechargez et prétraitez MNIST (flatten + normalisation).  
Créez ensuite une copie des labels d’entraînement et **mélangez-la aléatoirement** (shuffle) pour casser la relation image→label.

Construisez un modèle dense (512 ReLU → 10 Softmax), compilez-le avec :
- optimizer = `"rmsprop"`
- loss = `"sparse_categorical_crossentropy"`
- metric = `"accuracy"`

Puis entraînez-le sur les **labels mélangés** pendant 100 époques (batch_size=128, validation_split=0.2).

In [ ]:
(train_images, train_labels), _ = mnist.load_data()
train_images = train_images.reshape((60000, 28 * 28))
train_images = train_images.astype("float32") / 255

random_train_labels = train_labels[:]
np.random.shuffle(random_train_labels)

model = keras.Sequential(
    [
        layers.Dense(512, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ]
)
model.compile(
    optimizer="rmsprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.fit(
    train_images,
    random_train_labels,
    epochs=100,
    batch_size=128,
    validation_split=0.2,
)

En repartant de MNIST prétraité, entraînez un modèle Dense(512 ReLU → 10 Softmax) en utilisant l’optimiseur **RMSprop** avec un **learning rate très élevé** (par ex. 1.0).  
Entraînez 10 époques (batch_size=128, validation_split=0.2) et observez la stabilité de l’apprentissage.

In [ ]:
(train_images, train_labels), _ = mnist.load_data()
train_images = train_images.reshape((60000, 28 * 28))
train_images = train_images.astype("float32") / 255

model = keras.Sequential(
    [
        layers.Dense(512, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ]
)
model.compile(
    optimizer=keras.optimizers.RMSprop(learning_rate=1.0),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.fit(
    train_images, train_labels, epochs=10, batch_size=128, validation_split=0.2
)

Ré-entraînez le même modèle MNIST (même architecture) mais avec RMSprop et un **learning rate plus faible** (par ex. 1e-2).  
Comparez qualitativement l’apprentissage avec le cas précédent (lr=1.0).

In [ ]:
model = keras.Sequential(
    [
        layers.Dense(512, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ]
)
model.compile(
    optimizer=keras.optimizers.RMSprop(learning_rate=1e-2),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.fit(
    train_images, train_labels, epochs=10, batch_size=128, validation_split=0.2
)

Construisez un modèle **très simple** pour MNIST : une seule couche `Dense(10, activation="softmax")`.  
Compilez-le (optimizer="rmsprop", loss="sparse_categorical_crossentropy", metric="accuracy") puis entraînez-le 20 époques (batch_size=128, validation_split=0.2).  
Conservez l’historique sous un nom explicite (ex : `history_small_model`).

In [ ]:
model = keras.Sequential([layers.Dense(10, activation="softmax")])
model.compile(
    optimizer="rmsprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
history_small_model = model.fit(
    train_images, train_labels, epochs=20, batch_size=128, validation_split=0.2
)

À partir de `history_small_model`, récupérez la **validation loss** et tracez-la en fonction des 20 époques.  
Ajoutez titre, labels, légende et affichez la figure.

In [ ]:
import matplotlib.pyplot as plt

val_loss = history_small_model.history["val_loss"]
epochs = range(1, 21)
plt.plot(epochs, val_loss, "b-", label="Validation loss")
plt.title("Validation loss for a model with insufficient capacity")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

Construisez un modèle plus capacitaire pour MNIST :
- Dense(128, relu)
- Dense(128, relu)
- Dense(10, softmax)

Compilez (rmsprop + sparse_categorical_crossentropy + accuracy) puis entraînez 20 époques (batch_size=128, validation_split=0.2).  
Stockez l’historique (ex : `history_large_model`).

In [ ]:
model = keras.Sequential(
    [
        layers.Dense(128, activation="relu"),
        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ]
)
model.compile(
    optimizer="rmsprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
history_large_model = model.fit(
    train_images,
    train_labels,
    epochs=20,
    batch_size=128,
    validation_split=0.2,
)

Tracez la **validation loss** de `history_large_model` sur 20 époques (graphique similaire au cas précédent) pour comparer l’évolution avec le petit modèle.

In [ ]:
val_loss = history_large_model.history["val_loss"]
epochs = range(1, 21)
plt.plot(epochs, val_loss, "b-", label="Validation loss")
plt.title("Validation loss for a model with appropriate capacity")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

Construisez un modèle **très capacitaire** pour MNIST avec trois couches cachées :
- Dense(2048, relu) × 3
- Dense(10, softmax)

Compilez comme précédemment, puis entraînez 20 époques avec un batch_size plus petit (par ex. 32) et validation_split=0.2.  
Stockez l’historique (ex : `history_very_large_model`).

In [ ]:
model = keras.Sequential(
    [
        layers.Dense(2048, activation="relu"),
        layers.Dense(2048, activation="relu"),
        layers.Dense(2048, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ]
)
model.compile(
    optimizer="rmsprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
history_very_large_model = model.fit(
    train_images,
    train_labels,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
)

Tracez la **validation loss** du modèle très grand sur 20 époques afin d’illustrer le risque de surapprentissage lorsque la capacité est excessive.

In [ ]:
val_loss = history_very_large_model.history["val_loss"]
epochs = range(1, 21)
plt.plot(epochs, val_loss, "b-", label="Validation loss")
plt.title("Validation loss for a model with too much capacity")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

Chargez le dataset **IMDB** (en limitant le vocabulaire aux 10 000 mots les plus fréquents).  
Implémentez une fonction `vectorize_sequences` qui transforme chaque séquence d’indices en vecteur **one-hot** de dimension 10 000.

Entraînez ensuite un modèle de base pour la classification binaire :
- Dense(16, relu)
- Dense(16, relu)
- Dense(1, sigmoid)

Compilez (optimizer="rmsprop", loss="binary_crossentropy", metric="accuracy")  
Entraînez 20 époques (batch_size=512, validation_split=0.4) et stockez l’historique (`history_original`).

In [ ]:
from keras.datasets import imdb

(train_data, train_labels), _ = imdb.load_data(num_words=10000)

def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.0
    return results

train_data = vectorize_sequences(train_data)

model = keras.Sequential(
    [
        layers.Dense(16, activation="relu"),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ]
)
model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
history_original = model.fit(
    train_data,
    train_labels,
    epochs=20,
    batch_size=512,
    validation_split=0.4,
)

En gardant le même prétraitement IMDB, entraînez un modèle plus petit :
- Dense(4, relu)
- Dense(4, relu)
- Dense(1, sigmoid)

Même compilation et mêmes hyperparamètres d’entraînement (20 époques, batch_size=512, validation_split=0.4).  
Stockez l’historique (`history_smaller_model`).

In [ ]:
model = keras.Sequential(
    [
        layers.Dense(4, activation="relu"),
        layers.Dense(4, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ]
)
model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
history_smaller_model = model.fit(
    train_data,
    train_labels,
    epochs=20,
    batch_size=512,
    validation_split=0.4,
)

Tracez sur un même graphique la **validation loss** du modèle original (`history_original`) et du modèle plus petit (`history_smaller_model`) sur 20 époques, avec titre, axes, ticks et légende.

In [ ]:
original_val_loss = history_original.history["val_loss"]
smaller_model_val_loss = history_smaller_model.history["val_loss"]
epochs = range(1, 21)
plt.plot(
    epochs,
    original_val_loss,
    "r--",
    label="Validation loss of original model",
)
plt.plot(
    epochs,
    smaller_model_val_loss,
    "b-",
    label="Validation loss of smaller model",
)
plt.title("Original model vs. smaller model (IMDB review classification)")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.xticks(epochs)
plt.legend()
plt.show()

Toujours sur IMDB vectorisé, entraînez un modèle beaucoup plus grand :
- Dense(512, relu)
- Dense(512, relu)
- Dense(1, sigmoid)

Même compilation et mêmes paramètres d’entraînement (20 époques, batch_size=512, validation_split=0.4).  
Stockez l’historique (`history_larger_model`).

In [ ]:
model = keras.Sequential(
    [
        layers.Dense(512, activation="relu"),
        layers.Dense(512, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ]
)
model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
history_larger_model = model.fit(
    train_data,
    train_labels,
    epochs=20,
    batch_size=512,
    validation_split=0.4,
)

Tracez sur un même graphique la **validation loss** du modèle original (`history_original`) et du modèle plus grand (`history_larger_model`) pour illustrer l’effet d’une capacité excessive sur la généralisation.

In [ ]:
original_val_loss = history_original.history["val_loss"]
larger_model_val_loss = history_larger_model.history["val_loss"]
epochs = range(1, 21)
plt.plot(
    epochs,
    original_val_loss,
    "r--",
    label="Validation loss of original model",
)
plt.plot(
    epochs,
    larger_model_val_loss,
    "b-",
    label="Validation loss of larger model",
)
plt.title("Original model vs. larger model (IMDB review classification)")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.xticks(epochs)
plt.legend()
plt.show()

En apprentissage profond, la régularisation regroupe des techniques destinées à améliorer la **généralisation** du modèle, c’est-à-dire sa capacité à bien fonctionner sur des données nouvelles, et non seulement sur les données d’entraînement.

---

## 1. Régularisation des poids (L1, L2)

La régularisation des poids consiste à ajouter une pénalité lors de l’entraînement lorsque les poids du réseau deviennent trop grands.  
L’objectif est d’éviter que le modèle ne devienne trop complexe ou trop sensible à des détails spécifiques du jeu d’entraînement.

### L2 (weight decay)

- Encourage les poids à rester petits.
- Rend le modèle plus stable et plus lisse.
- Réduit le risque de surapprentissage.
- N’annule généralement pas complètement les poids.

C’est la forme de régularisation la plus couramment utilisée.

---

### L1

- Encourage certains poids à devenir exactement nuls.
- Introduit une forme de sélection automatique de variables.
- Produit des modèles plus simples et plus parcimonieux.

Elle est utile lorsque l’on soupçonne que seules certaines caractéristiques sont réellement pertinentes.

---

### L1 + L2

- Combine les avantages des deux approches.
- Permet à la fois de limiter la taille des poids et d’en annuler certains.

---

## 2. Dropout

Le dropout est une technique différente :  
au lieu de pénaliser les poids, il modifie temporairement la structure du réseau pendant l’entraînement.

Concrètement :

- À chaque itération, une proportion aléatoire de neurones est désactivée.
- Le réseau ne peut donc pas s’appuyer toujours sur les mêmes neurones.
- Cela empêche les neurones de trop se spécialiser ou de se co-adapter.

Effets principaux :

- Apprentissage de représentations plus robustes.
- Réduction du surapprentissage.
- Effet comparable à une moyenne implicite de plusieurs sous-modèles.

Important :
- Le dropout est actif uniquement pendant l’entraînement.
- Il est automatiquement désactivé lors de la validation ou du test.

---

## 3. Différence essentielle

- La régularisation L1/L2 agit sur les **valeurs des poids**.
- Le dropout agit sur la **structure temporaire du réseau pendant l’entraînement**.

Les deux techniques peuvent être combinées et sont souvent utilisées ensemble pour améliorer la généralisation d’un modèle.

Modifiez le modèle IMDB (16-16-1) pour ajouter une régularisation **L2** sur les poids des couches denses (par ex. `l2(0.002)` via `kernel_regularizer`).  
Entraînez le modèle (mêmes hyperparamètres) et stockez l’historique (`history_l2_reg`).

In [ ]:
from keras.regularizers import l2

model = keras.Sequential(
    [
        layers.Dense(16, kernel_regularizer=l2(0.002), activation="relu"),
        layers.Dense(16, kernel_regularizer=l2(0.002), activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ]
)
model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
history_l2_reg = model.fit(
    train_data,
    train_labels,
    epochs=20,
    batch_size=512,
    validation_split=0.4,
)

Tracez sur un même graphique la **validation loss** du modèle original (`history_original`) et du modèle avec **L2** (`history_l2_reg`) sur 20 époques (titre + légende).

In [ ]:
original_val_loss = history_original.history["val_loss"]
l2_val_loss = history_l2_reg.history["val_loss"]
epochs = range(1, 21)
plt.plot(
    epochs,
    original_val_loss,
    "r--",
    label="Validation loss of original model",
)
plt.plot(
    epochs,
    l2_val_loss,
    "b-",
    label="Validation loss of L2-regularized model",
)
plt.title(
    "Original model vs. L2-regularized model (IMDB review classification)"
)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.xticks(epochs)
plt.legend()
plt.show()

Importez `keras.regularizers` puis montrez comment créer :
- une régularisation L1 (ex : coefficient 0.001),
- une régularisation combinée L1+L2 (ex : 0.001 et 0.001).

In [ ]:
from keras import regularizers

regularizers.l1(0.001)
regularizers.l1_l2(l1=0.001, l2=0.001)

Construisez une variante du modèle IMDB (16-16-1) en ajoutant du **Dropout** après chaque couche Dense cachée (par ex. `Dropout(0.5)`).  
Compilez et entraînez avec les mêmes paramètres (20 époques, batch_size=512, validation_split=0.4).  
Stockez l’historique (`history_dropout`).

In [ ]:
model = keras.Sequential(
    [
        layers.Dense(16, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(16, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(1, activation="sigmoid"),
    ]
)
model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
history_dropout = model.fit(
    train_data,
    train_labels,
    epochs=20,
    batch_size=512,
    validation_split=0.4,
)

In [ ]:
original_val_loss = history_original.history["val_loss"]
dropout_val_loss = history_dropout.history["val_loss"]
epochs = range(1, 21)
plt.plot(
    epochs,
    original_val_loss,
    "r--",
    label="Validation loss of original model",
)
plt.plot(
    epochs,
    dropout_val_loss,
    "b-",
    label="Validation loss of dropout-regularized model",
)
plt.title(
    "Original model vs. dropout-regularized model (IMDB review classification)"
)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.xticks(epochs)
plt.legend()
plt.show()